In [ ]:
from aicsimageio import AICSImage
import dask_image.imread


In [ ]:
path = "/Volumes/KINGSTON/code/phd/image-analysis/synapse-counting/test-images-VLGUT1-PSD95-A/OE_Exp1_IHC_Exp1_HA-GPR37L1_555-VGLUT1_647-PSD95_63X_airyscan_1.8zoom_CA1_SO.czi"

img = AICSImage(path)
img.dask_data 

In [ ]:
import os
import pandas as pd
import dask

from metadata import extract_metadata, image_filename
from preprocessing import extract_and_split, ImagePreprocessing
from calc_synaptic_coloc import pearsons_coloc


In [ ]:
input_folder = "/Volumes/KINGSTON/code/phd/image-analysis/synapse-counting/test-images-VLGUT1-PSD95-A"

In [ ]:
@delayed
def process_image_pearson(filename):    
    if filename.endswith(".czi"):
        # getting the filepath
        file_path = os.path.join(input_folder, filename)
        # preprocessing
        pre, post = extract_and_split(file_path)
        p = ImagePreprocessing(include_rolling_ball=True, include_blur=True, include_clahe=True, include_tophat=True)
        pre_1, post_1 = p.preprocess(pre, post)
        # getting the data
        pearson_cor, pvalue, pearson_cor_rot, pvalue_rot = pearsons_coloc(pre_1, post_1)
        # getting the right filename
        img_filename = image_filename(filename, [1,2,3,4,11]) 
        return {
            "img_filename": img_filename,
            "pearson_cor": pearson_cor,
            "pvalue": pvalue,
            "pearson_cor_rot": pearson_cor_rot,
            "pvalue_rot": pvalue_rot
        }
    else:
        return None

In [ ]:
# get a list of files in that input_folder
file_list = os.listdir(input_folder)

In [ ]:
delayed_results = [process_image_pearson(filename) for filename in file_list]

In [ ]:
dask.visualize(*delayed_results)

In [ ]:
results = dask.compute(*delayed_results)

In [ ]:
df = pd.DataFrame(results)
df.head(10)

In [ ]:
# get a list of files in that input_folder
file_list = os.listdir(input_folder)

# empty dict to store the results
results = []

# the actual function
for filename in file_list:
    if filename.endswith(".czi"):
        
        # getting the filepath
        file_path = os.path.join(input_folder, filename)
        
        # metadata
        pixel_size_um, image_size_pix, image_size_um = extract_metadata(file_path)
        
        # preprocessing
        pre, post = extract_and_split(file_path)
        p = ImagePreprocessing(include_rolling_ball=True, include_blur=True, include_clahe=True, include_tophat=True)
        pre_1, post_1 = delayed(p.preprocess)(pre, post)
        
        # getting the data
        pearson_cor, pvalue, pearson_cor_rot, pvalue_rot = delayed(pearsons_coloc)(pre_1, post_1)
        
        # getting the right filename
        img_filename = delayed(image_filename)(filename, [1,2,3,4,11]) 

        results.append({
            "img_filename": img_filename,
            "pearson_cor": pearson_cor,
            "pvalue": pearson_cor_rot,
            "pearson_cor_rot": pearson_cor_rot,
            "pvalue_rot": pvalue_rot
        })

In [ ]:
df.head(30)